In [1]:
from typing_extensions import TypedDict, Annotated
from langgraph.types import Send
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from operator import add

# llm = init_chat_model(model="llama3.1:8B", model_provider="ollama")
llm = init_chat_model(model="gpt-4o-mini", model_provider="openai")

In [2]:
class State(TypedDict):
    document: str
    final_summary: str
    summaries: Annotated[list[dict], add]

In [ ]:
def summarize_p(args):
    paragraph = args["paragraph"]
    index = args["index"]

    response = llm.invoke(
        f"Write a 3-sentence summary for this paragraph: {paragraph}",
    )

    return {"summaries": [{"summary": response.content, "index": index}]}


def dispatch_summarizers(state: State):
    chunks = state["document"].split("\n\n")
    return [
        Send("summarize_p", {"paragraph": chunk, "index": index})
        for index, chunk in enumerate(chunks)
    ]


def final_summary(state: State):
    response = llm.invoke(
        f"Using the following summaries, give me a final one {state["summaries"]}"
    )

    return {
        "final_summary": response.content,
    }

In [4]:
graph_builder = StateGraph(State)

graph_builder.add_node("summarize_p", summarize_p)
graph_builder.add_node("final_summary", final_summary)

graph_builder.add_conditional_edges(START, dispatch_summarizers, ["summarize_p"])
graph_builder.add_edge("summarize_p", "final_summary")
graph_builder.add_edge("final_summary", END)

graph = graph_builder.compile()

# graph

In [5]:
with open("fed_transcript.md", "r", encoding="utf-8") as file:
    document = file.read()
    for chunk in graph.stream(
        {"document": document},
        stream_mode="updates", # 어떤 노드가 업데이트 했는지 볼 수 있음.
    ):
        print(chunk, "\n")
        

{'summarize_p': {'summaries': [{'summary': 'In August, the unemployment rate increased slightly to 4.3 percent. Despite this uptick, the rate has remained largely stable over the past year. Overall, the unemployment rate continues to reflect a relatively low level in the labor market.', 'index': 5}]}} 

{'summarize_p': {'summaries': [{'summary': 'The moderation in economic activity is primarily attributed to a decrease in consumer spending. This slowdown has significant implications for overall growth. As consumer behavior shifts, it impacts various sectors of the economy.', 'index': 2}]}} 

{'summarize_p': {'summaries': [{'summary': "Business investment in equipment and intangible assets has shown improvement compared to the previous year's levels. This increase indicates a positive shift in companies' expenditure on growth-oriented resources. Overall, the trend suggests a more optimistic outlook for business development and capital investment.", 'index': 3}]}} 

{'summarize_p': {'sum